# Sprint 6 — 03 Model Serving Deployment

**Objective:** deploy registered model `workspace.gcivilization.civ4_rag_model` (version 1) as a **Databricks Model Serving** endpoint, and give the served model secure credentials to call the two external Databricks services it depends on: **Vector Search** (retrieval) and the **Foundation Model APIs** (embeddings + LLM).

## Architecture

* **Endpoint:** `civ4_rag_endpoint` — serverless CPU (`Small` workload, **scale-to-zero** enabled for cost control) serving the Unity Catalog model version. Free Edition supports custom model serving endpoints on CPU (with caps on active endpoints; no GPU / provisioned throughput).
* **Credentials:** Model Serving containers have **no ambient workspace credentials**. The supported mechanism is **secret-based environment variables** on the served entity:
  * `DATABRICKS_HOST` — plain env var with the workspace URL (public value).
  * `DATABRICKS_TOKEN` — backed by `{{secrets/civ4_rag_serving/model_serving_token}}`, never stored in plaintext.
  * Both the `databricks-sdk` `WorkspaceClient` and `mlflow.deployments` client (the two clients `rag_utils` creates) authenticate from these variables.
* **Token lifecycle:** the token is a 30-day personal access token created via the Tokens API and stored directly in the secret scope — **the token value is never printed, logged, or written to any artifact**. On expiry, re-run this notebook to mint and store a fresh token (rotation is idempotent; old tokens with the same comment are revoked).

## Prerequisites

* `02_model_registration` completed (registered model version READY).
* User approval for programmatic token + secret-scope creation (obtained for this sprint).

## Inputs / outputs

* **Inputs:** registered model version, workspace host URL.
* **Outputs:** serving endpoint `civ4_rag_endpoint` in READY state + secret scope `civ4_rag_serving`.

In [0]:
# Setup — deployment constants and the target model version.
import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
w = WorkspaceClient()

REGISTERED_MODEL_NAME = "workspace.gcivilization.civ4_rag_model"
ENDPOINT_NAME = "civ4_rag_endpoint"
SECRET_SCOPE = "civ4_rag_serving"
SECRET_KEY = "model_serving_token"
TOKEN_COMMENT = "civ4-rag model serving (Sprint 6)"
TOKEN_LIFETIME_SECONDS = 30 * 24 * 3600  # 30 days — rotate by re-running this notebook
HOST = w.config.host

# Deploy the latest registered version (traceable to notebook 02's MLflow run).
# NOTE: UC registry does not support get_latest_versions(); use version search +
# the recommended alias pattern instead.
client = MlflowClient(registry_uri="databricks-uc")
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
MODEL_VERSION = str(max(int(v.version) for v in versions))
try:
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", MODEL_VERSION)
    print(f"PASS  alias 'champion' -> version {MODEL_VERSION} (recommended UC deployment pattern)")
except Exception as e:
    print(f"NOTE  alias not set ({type(e).__name__}) — deploying by explicit version instead")

print(f"Endpoint        : {ENDPOINT_NAME}")
print(f"Served model    : {REGISTERED_MODEL_NAME} (version {MODEL_VERSION})")
print(f"Workspace host  : {HOST}")
print(f"Secret scope    : {SECRET_SCOPE} / key {SECRET_KEY}")
assert MODEL_VERSION == "1" or MODEL_VERSION, MODEL_VERSION
print("PASS  deployment targets configured")

In [0]:
# Credentials — secret scope + token provisioning (approved by the user).
# SECURITY: the token value is created and written straight into the secret
# store; it is never printed, logged, or persisted anywhere else.

# 1. Secret scope (create once)
scopes = [s.name for s in w.secrets.list_scopes()]
if SECRET_SCOPE not in scopes:
    w.secrets.create_scope(scope=SECRET_SCOPE)
    print(f"PASS  secret scope '{SECRET_SCOPE}' created")
else:
    print(f"PASS  secret scope '{SECRET_SCOPE}' already exists (reused)")

# 2. Mint a fresh 30-day token and store it in the scope
new_token = w.tokens.create(comment=TOKEN_COMMENT, lifetime_seconds=TOKEN_LIFETIME_SECONDS)
new_token_id = getattr(getattr(new_token, "token_info", None), "token_id", None) or getattr(new_token, "token_id", None)
w.secrets.put_secret(scope=SECRET_SCOPE, key=SECRET_KEY, string_value=new_token.token_value)

del new_token  # drop the in-memory reference immediately

# 3. Rotation hygiene: revoke older tokens with the same comment (best effort)
revoked = 0
for t in w.tokens.list():
    tid = getattr(t, "token_id", None)
    if tid and tid != new_token_id and getattr(t, "comment", "") == TOKEN_COMMENT:
        try:
            w.tokens.delete(token_id=tid)
            revoked += 1
        except Exception:
            pass
if revoked:
    print(f"PASS  rotated: {revoked} old token(s) with the same comment revoked")

# 4. Verify the secret key exists (metadata only — value is never read here)
keys = [s.key for s in w.secrets.list_secrets(scope=SECRET_SCOPE)]
assert SECRET_KEY in keys, f"{SECRET_KEY} not found in scope {SECRET_SCOPE}"
print(f"PASS  token stored as secret {SECRET_SCOPE}/{SECRET_KEY} (value never displayed)")

In [0]:
# Create (or update) the serving endpoint — CPU Small, scale-to-zero,
# serving the registered UC model version, with secret-backed env vars.
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput

ENV_VARS = {
    "DATABRICKS_HOST": HOST,                                   # public value, plain env var
    "DATABRICKS_TOKEN": f"{{{{secrets/{SECRET_SCOPE}/{SECRET_KEY}}}}}",  # secret-backed, never plaintext
}

served_entity = ServedEntityInput(
    entity_name=REGISTERED_MODEL_NAME,
    entity_version=MODEL_VERSION,
    workload_size="Small",            # serverless CPU (Free Edition: no GPU serving)
    scale_to_zero_enabled=True,       # cost control for a portfolio project
    environment_vars=ENV_VARS,
)

existing = {ep.name for ep in w.serving_endpoints.list()}
if ENDPOINT_NAME in existing:
    w.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_entities=[served_entity],
    )
    print(f"PASS  endpoint '{ENDPOINT_NAME}' already existed — config updated (new version rollout)")
else:
    w.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(name=ENDPOINT_NAME, served_entities=[served_entity]),
        description=(
            "Civ4 RAG assistant — Sprint 5 Config B (top_k=10, prompt v1, temp 0.1). "
            "PyFunc orchestrating Vector Search + Foundation Model APIs."
        ),
    )
    print(f"PASS  endpoint '{ENDPOINT_NAME}' created — serving {REGISTERED_MODEL_NAME} v{MODEL_VERSION}")

In [0]:
# Wait for readiness — first deployment builds the serving environment
# (image build + pip install of the model's requirements), typically 5-15 min.
# This cell polls for up to 6 minutes per run; RE-RUN it while the endpoint is
# still provisioning (it is safe and stateless).
import time

DEADLINE_SECONDS = 6 * 60
POLL_INTERVAL = 30

start = time.time()
ready = None
while time.time() - start < DEADLINE_SECONDS:
    ep = w.serving_endpoints.get(ENDPOINT_NAME)
    ready = getattr(ep.state.ready, "value", ep.state.ready)
    config_update = getattr(ep.state.config_update, "value", ep.state.config_update)
    print(f"  [{int(time.time() - start):>3}s] state={ready} config_update={config_update}")
    if ready == "READY":
        break
    if config_update in ("UPDATE_FAILED", "UPDATE_CANCELED"):
        raise RuntimeError(f"Endpoint update failed: {ep.state}")
    time.sleep(POLL_INTERVAL)

if ready == "READY":
    print("PASS  endpoint is READY")
else:
    print(f"Still provisioning (state={ready}) — re-run this cell to continue polling.")

In [0]:
# Verify the deployed configuration — served entity, workload, and (safely)
# that the token env var is SECRET-BACKED, not plaintext.
ep = w.serving_endpoints.get(ENDPOINT_NAME)
ready = getattr(ep.state.ready, "value", ep.state.ready)
assert ready == "READY", f"endpoint state: {ready}"

se = ep.config.served_entities[0]
assert se.entity_name == REGISTERED_MODEL_NAME
assert se.entity_version == MODEL_VERSION
assert se.workload_size == "Small"
assert se.scale_to_zero_enabled is True

env_vars = se.environment_vars or {}
assert env_vars.get("DATABRICKS_HOST") == HOST
assert env_vars.get("DATABRICKS_TOKEN", "").startswith("{{secrets/"), \
    "DATABRICKS_TOKEN must be secret-backed, not plaintext!"

print(f"Endpoint        : {ep.name} ({ready})")
print(f"Served model    : {se.entity_name} version {se.entity_version}")
print(f"Workload        : {se.workload_size} (CPU), scale_to_zero={se.scale_to_zero_enabled}")
print(f"Config version  : {ep.config.config_version}")
print(f"Env var DATABRICKS_HOST   : {env_vars['DATABRICKS_HOST']} (plain, public)")
print(f"Env var DATABRICKS_TOKEN : {env_vars['DATABRICKS_TOKEN']} (secret reference — value never exposed)")
print("PASS  endpoint configuration verified")

In [0]:
# Summary — Sprint 6 deployment acceptance.
print("=" * 60)
print("SPRINT 6 NOTEBOOK 03: DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"""
Endpoint          : {ENDPOINT_NAME} (READY)
Served model      : {REGISTERED_MODEL_NAME} version {MODEL_VERSION}
Compute            : serverless CPU 'Small', scale-to-zero
Credentials        : DATABRICKS_HOST (plain env var) + DATABRICKS_TOKEN ({{secrets/{SECRET_SCOPE}/{SECRET_KEY}}})
Token lifetime     : 30 days — rotate by re-running the credentials cell of this notebook
External services : Vector Search (civ4_vs_endpoint) + Foundation Model APIs

Manual follow-ups (documented in README.md):
- The PAT expires after 30 days; re-run this notebook to rotate it.
- Free Edition caps the number of active serving endpoints; delete unused ones.
""")
print("READY for 04_inference_testing")